# 00 — Run the whole project on Colab (L4 GPU)

Run the cells **top to bottom**. Everything important is saved to Google Drive
(`MyDrive/DL-Project/`), so if Colab disconnects, just run all cells again: finished
steps are skipped and training resumes from the last completed epoch.

**Before you start:** Runtime → Change runtime type → **L4 GPU**.

| Step | What it does | Time on L4 (approx.) |
|---|---|---|
| 1–3 | GPU check, Drive, code | 1 min |
| 4 | Download Flickr8k (first time only) | 2 min |
| 5 | Splits + vocabulary (first time only) | < 1 min |
| 6 | ResNet50 features (first time only) | 5–10 min |
| 7 | Smoke test | 1 min |
| 8 | Core frozen runs: 3 decoders × 3 seeds | ~30–60 min |
| 9 | Fine-tuned runs: 3 decoders × 3 seeds | ~1–2 h |
| 10 | Paper-protocol replication | ~10 min |
| 11 | Results tables | < 1 min |

Times are estimates until the first real runs; check the Resources panel for units used.

## 1. Check the GPU

In [6]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
assert gpus, "No GPU! Runtime → Change runtime type → L4 GPU, then run again."
print("TensorFlow", tf.__version__, "| GPU:", gpus)

name, memory.total [MiB]
NVIDIA L4, 23034 MiB
TensorFlow 2.21.0 | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Mount Google Drive (everything persistent lives here)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/DL-Project'          # persistent: splits, features, runs, results
LOCAL = '/content/data_local'                         # fast local disk: raw images, feature copies
for d in [f'{DRIVE}/processed', f'{DRIVE}/features', f'{DRIVE}/runs', f'{DRIVE}/results',
          f'{LOCAL}/raw', f'{LOCAL}/features']:
    os.makedirs(d, exist_ok=True)
print("Drive folder:", DRIVE)

Mounted at /content/drive
Drive folder: /content/drive/MyDrive/DL-Project


## 3. Get the code

`BRANCH` is the branch with the new pipeline. After it is merged into `main`, change it to `main`.

In [3]:
REPO = '/content/DL-Project'
BRANCH = 'day1-pipeline'
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} https://github.com/arz1742/DL-Project.git {REPO}
else:
    !git -C {REPO} fetch -q origin {BRANCH} && git -C {REPO} checkout -q {BRANCH} && git -C {REPO} pull -q
%cd {REPO}
!git log --oneline -1
!pip install -q pycocoevalcap

# Point the repo's data/ folders at Drive (persistent) and local disk (fast)
!rm -rf data/processed data/features runs results
!ln -s {DRIVE}/processed data/processed
!ln -s {LOCAL}/features  data/features
!ln -s {DRIVE}/runs      runs
!ln -s {DRIVE}/results   results
!java -version 2>&1 | head -1 || echo "No Java: METEOR will use the NLTK version"

/content/DL-Project
167dcce (HEAD -> day1-pipeline, origin/day1-pipeline) Created using Colab
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 25.7 MB/s eta 0:00:00
[0.021s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


## 4. Download Flickr8k (first time only)

Needs your Kaggle API key. Either add Colab secrets named `KAGGLE_USERNAME` and `KAGGLE_KEY`
(key icon in the left sidebar), or upload `kaggle.json` when asked. The zip is cached on Drive.

In [4]:
ZIP = f'{DRIVE}/flickr8k.zip'
if not os.path.exists(ZIP):
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    except Exception:
        from google.colab import files
        print("Upload kaggle.json"); files.upload()
        !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d adityajn105/flickr8k -p {DRIVE}
    !mv {DRIVE}/flickr8k.zip {ZIP} 2>/dev/null || true
if not os.path.exists(f'{LOCAL}/raw/Images'):
    !unzip -q -o {ZIP} -d {LOCAL}/raw
print(len(os.listdir(f'{LOCAL}/raw/Images')), "images |", f'{LOCAL}/raw/captions.txt')

8091 images | /content/data_local/raw/captions.txt


## 5. Splits (6091 / 1000 / 1000 images) and vocabulary — first time only

In [7]:
if not os.path.exists('data/processed/tokenizer.json'):
    !python src/data_loader.py --raw_dir {LOCAL}/raw --out_dir data/processed
    !python build_tokenizer.py
else:
    print("Splits and tokenizer already on Drive, reusing them (keeps every run comparable).")
import json
for s in ['train', 'val', 'test']:
    print(s, len(json.load(open(f'data/processed/{s}_captions.json'))), 'images')

Splits and tokenizer already on Drive, reusing them (keeps every run comparable).
train 6091 images
val 1000 images
test 1000 images


## 6. ResNet50 features — first time only

`pooled` (2048-d, for frozen configs) and `conv4` (14×14×1024 maps, for fine-tuning, ~3.3 GB).
Saved to Drive once; later sessions copy them to the fast local disk.

In [8]:
for mode in ['pooled', 'conv4']:
    local, remote = f'data/features/resnet50_{mode}.npy', f'{DRIVE}/features/resnet50_{mode}.npy'
    if os.path.exists(local):
        continue
    if os.path.exists(remote):
        !cp {remote} {local} && cp {DRIVE}/features/image_index.json data/features/
    else:
        !python src/feature_extraction.py --mode {mode} --image_dir {LOCAL}/raw/Images
        !cp {local} {remote} && cp data/features/image_index.json {DRIVE}/features/
!ls -lh data/features

lrwxrwxrwx 1 root root 28 Oct 20 16:03 data/features -> /content/data_local/features


## 7. Smoke test (1 epoch on 200 images, thrown away afterwards)

In [7]:
!python train.py --config experiments/config_lstm_frozen.yaml --seed 0 --epochs 1 --limit_images 200 --runs_dir /content/smoke_runs --restart

I0000 00:00:1792488473.211517    5239 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792488475.600377    5239 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=== lstm_frozen | seed 0 | lstm decoder | frozen encoder ===
Train: 200 images / 1000 captions | Val: 40 images / 200 captions | vocab 2588
W0000 00:00:1792488477.728492    5239 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
  epoch 1: val_loss=5.2721 (best 5.2721 @ epoch 1, wait 0)
32/32 - 5s - 150ms/step - loss: 6.0092 - maske

## 8. Core frozen runs — LSTM, GRU, hybrid × seeds 1, 2, 3

Early stopping ends each run when validation loss stops improving for 5 epochs.

In [10]:
!python scripts/run_experiments.py --stage frozen --seeds 1,2,3


$ /usr/bin/python3 evaluate.py --config_name lstm_frozen --seed 1 --runs_dir runs --decodings greedy,beam3,beam5
I0000 00:00:1792490745.802837   17223 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792490748.107727   17223 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0000 00:00:1792490750.309772   17223 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
=== Evaluating lstm_frozen seed 1 on 1000 test images ===
Token metrics: {'loss': 2.7733, 'masked_acc': 0.4066, 'top5_acc': 0.6876,

## 9. Fine-tuned runs — start from each frozen model, then train ResNet50's last block too

Needs step 8 finished for the same seeds.

In [13]:
!python scripts/run_experiments.py --stage finetuned --seeds 1,2,3


$ /usr/bin/python3 train.py --config experiments/config_lstm_finetuned.yaml --seed 1 --runs_dir runs
I0000 00:00:1792493792.306050   46571 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792493794.713818   46571 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=== lstm_finetuned | seed 1 | lstm decoder | finetune encoder ===
Train: 6091 images / 30455 captions | Val: 1000 images / 5000 captions | vocab 2588
W0000 00:00:1792493797.233585   46571 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 

## 10. Paper-protocol replication (for the accuracy-illusion table)

Hybrid decoder trained exactly like Kavitha & Karpagam (2025): padding counted, 40 epochs, no early stopping.

In [9]:
!python scripts/run_experiments.py --stage paper --seeds 1


$ /usr/bin/python3 train.py --config experiments/config_paper_protocol.yaml --seed 1 --runs_dir runs
I0000 00:00:1792512466.247019    7862 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792512468.594642    7862 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=== paper_protocol | seed 1 | hybrid decoder | frozen encoder ===
Train: 6091 images / 30455 captions | Val: 1000 images / 5000 captions | vocab 2588
W0000 00:00:1792512471.085178    7862 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 

## 11. Results tables

In [10]:
!python scripts/aggregate_results.py

# Table A: Ablation (test set, beam5, mean ± std over seeds)

| Config | Seeds | bleu1 | bleu4 | meteor | rouge l | cider | masked acc | key object recall |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| C1 LSTM, frozen | 3 | 0.5810 ± 0.0061 | 0.1681 ± 0.0058 | 0.1751 ± 0.0035 | 0.4219 ± 0.0054 | 0.3772 ± 0.0241 | 0.4068 ± 0.0024 | 0.6082 ± 0.0109 |
| C2 GRU, frozen | 3 | 0.5803 ± 0.0084 | 0.1725 ± 0.0013 | 0.1792 ± 0.0021 | 0.4259 ± 0.0040 | 0.3951 ± 0.0059 | 0.4089 ± 0.0016 | 0.6197 ± 0.0078 |
| C3 Hybrid LSTM→GRU, frozen | 3 | 0.5857 ± 0.0145 | 0.1737 ± 0.0038 | 0.1789 ± 0.0003 | 0.4285 ± 0.0013 | 0.4013 ± 0.0061 | 0.4086 ± 0.0003 | 0.6147 ± 0.0070 |
| C4 LSTM, fine-tuned | 3 | 0.5849 ± 0.0068 | 0.1715 ± 0.0037 | 0.1788 ± 0.0020 | 0.4270 ± 0.0060 | 0.3923 ± 0.0155 | 0.4093 ± 0.0018 | 0.6161 ± 0.0097 |
| C5 GRU, fine-tuned | 3 | 0.5902 ± 0.0103 | 0.1791 ± 0.0015 | 0.1830 ± 0.0013 | 0.4350 ± 0.0012 | 0.4144 ± 0.0036 | 0.4132 ± 0.0016 | 0.6302 ± 0.0098 |
| C6 Hybrid LSTM→GRU

## 12. Save compute units

Disconnects the runtime so idle time doesn't use units. Run it whenever you're done.

In [11]:
from google.colab import runtime
runtime.unassign()